In [ ]:
import pickle, os, torch, numpy as np, pandas as pd, Levenshtein, matplotlib.pyplot as plt
from collections import Counter 
from sklearn.metrics import f1_score
from tqdm import tqdm
from utils import calculate_similarity, generate_hashid, three_to_one, pdb2proteinSeq, pdb2smile, prot2qt, lig2qt
from utils import extract_sequence_from_pdb, get_residues

In [ ]:
dfs = {}
for x in os.listdir():
    if x.endswith("pickle"):
        with open (""+x, "rb") as f:
            logs = pickle.load(f)
            true = list([int(x.item()) for x in logs['y']])
            pred = [1 if x.item()>0.5 else 0 for x in logs['scores'].sigmoid()]
            seqs = logs['seqs']
            smiles = logs['smiles']
            drug_ids = logs['drug_ids']
            prot_ids = logs['prot_ids']
            pdbs = logs["pdbs"]
            df = pd.DataFrame( {y:x for (x,y) in zip([true,pred,seqs,smiles, drug_ids,prot_ids,pdbs],["label", "pred", "seq", "smile","drug_id", "prot_id", "pdb"])} )
            dfs[x] = {}
            dfs[x]['df'] = df
            dfs[x]['f1'] = logs['f1']
            dfs[x]['auc'] = logs['auc']
            dfs[x]['auprc'] = logs['auprc']
            dfs[x]['conf_mat'] = logs['conf_mat']
            

In [ ]:
dataset = 'bind' # or 'drug'
default = dfs['default.default.' + dataset + '.pickle']['df']
furthest = dfs['default.furthest.' + dataset + '.pickle']['df']

In [ ]:
def_suf = default.add_suffix('_def')
fur_sif = furthest.add_suffix('_fur')
merged = pd.concat([def_suf, fur_sif], axis=1)
mismatch = merged[merged["pred_def"] != merged["pred_fur"]]
match = merged[merged["pred_def"] == merged["pred_fur"]]
non_empty = mismatch[mismatch["pdb_def"].str.strip() != ""]
mismatch.shape, match.shape

In [ ]:
import re

def parse_TM(output_text):
    tm1 = re.search(r'TM-score=\s+([\d.]+) \(if normalized by length of Chain_1\)', output_text)
    tm2 = re.search(r'TM-score=\s+([\d.]+) \(if normalized by length of Chain_2\)', output_text)
    rmsd = re.search(r'RMSD=\s+([\d.]+)', output_text)
    tm1 = float(tm1.group(1)) if tm1 else None
    tm2 = float(tm2.group(1)) if tm2 else None
    rmsd = float(rmsd.group(1)) if rmsd else None
    return tm1,tm2,rmsd
import subprocess
def run_tmalign(pdb1_path, pdb2_path, tmalign_path='TMalign'):
    try:
        result = subprocess.run([tmalign_path, pdb1_path, pdb2_path], capture_output=True, text=True)
        return result.stdout
    except Exception as e:
        return f"Error running TMalign: {e}"

In [ ]:
import subprocess,re
rmsd = []
tm1 = []
tm2 = []
rmsd_A = []
tm1_A = []
tm2_A = []

rmsd_mis=[]
tm1_mis = []
tm2_mis=[]
rmsd_mis_A=[]
tm1_mis_A = []
tm2_mis_A=[]
annn = 0
for i,row in match.iterrows():
    seq = row['seq_def']
    lig = row['smile_def']
    fur_seq = row['seq_fur']
    pred_def = row["pred_def"]
    pred_fur = row["pred_fur"]
    hash = f"{generate_hashid(seq)}_{generate_hashid(lig)}"
    althash = f"{generate_hashid(fur_seq)}_{generate_hashid(lig)}"
    path = # path to 3d complex for representative-ligand pair generated by diffdock
    altpath = # path to 3d complex for variant-ligand pair generated by diffdock
    
    if os.path.isfile(path) and os.path.isfile(altpath):
        t1,t2,rm = parse_TM(run_tmalign(path,altpath))
        tm1.append(t1)
        tm2.append(t2)
        rmsd.append(rm)
    
    path = # path to 3d complex for representative-ligand pair generated by alphafold3 
    altpath = # path to 3d complex for variant-ligand pair generated by alphafold3 

    if os.path.isfile(path) and os.path.isfile(altpath):
        t1,t2,rm = parse_TM(run_tmalign(path,altpath))
        tm1_A.append(t1)
        tm2_A.append(t2)
        rmsd_A.append(rm)

for i,row in mismatch.iterrows():
    seq = row['seq_def']
    lig = row['smile_def']
    fur_seq = row['seq_fur']
    pred_def = row["pred_def"]
    pred_fur = row["pred_fur"]
    hash = f"{generate_hashid(seq)}_{generate_hashid(lig)}"
    althash = f"{generate_hashid(fur_seq)}_{generate_hashid(lig)}"
    path = # path to 3d complex for representative-ligand pair generated by diffdock
    altpath = # path to 3d complex for variant-ligand pair generated by diffdock
    
    if os.path.isfile(path) and os.path.isfile(altpath):
        t1,t2,rm = parse_TM(run_tmalign(path,altpath))
        tm1_mis.append(t1)
        tm2_mis.append(t2)
        rmsd_mis.append(rm)

    path = # path to 3d complex for representative-ligand pair generated by diffdock
    altpath = # path to 3d complex for variant-ligand pair generated by diffdock
    if os.path.isfile(path) and os.path.isfile(altpath):
        t1,t2,rm = parse_TM(run_tmalign(path,altpath))
        tm1_mis_A.append(t1)
        tm2_mis_A.append(t2)
        rmsd_mis_A.append(rm)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

sns.set(style="whitegrid", font_scale=1.1)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharey=True)

###################################
# ---- First Pair (rmsd_A) ----
###################################

df1 = pd.DataFrame({
    "RMSD": np.concatenate([rmsd_A, rmsd_mis_A]),
    "Group": ["Matched"] * len(rmsd_A) + ["Mismatched"] * len(rmsd_mis_A)
})

sns.violinplot(
    x="Group",
    y="RMSD",
    data=df1,
    inner=None,
    cut=0,
    ax=axes[0]
)

sns.boxplot(
    x="Group",
    y="RMSD",
    data=df,
    width=0.02,
    showcaps=True,
    boxprops={
        "facecolor": "black",
        "edgecolor": "black",
        "zorder": 3
    },
    medianprops={
        "color": "white",
        "linewidth": 2
    },
    whiskerprops={"color": "black"},
    capprops={"color": "black"},
    showfliers=False,
    ax=axes[0]
)

axes[0].set_xlabel("")
axes[0].set_ylabel("RMSD")
axes[0].set_title("RMSD Distribution Comparison (AlphaFold3)")

stat1, p1 = mannwhitneyu(rmsd_A, rmsd_mis_A, alternative="two-sided")


###################################
# ---- Second Pair (rmsd) ----
###################################

df2 = pd.DataFrame({
    "RMSD": np.concatenate([rmsd, rmsd_mis]),
    "Group": ["Matched"] * len(rmsd) + ["Mismatched"] * len(rmsd_mis)
})

sns.violinplot(
    x="Group",
    y="RMSD",
    data=df2,
    inner=None,
    cut=0,
    ax=axes[1]
)

sns.boxplot(
    x="Group",
    y="RMSD",
    data=df2,
    width=0.02,
    showcaps=True,
    boxprops={
        "facecolor": "black",
        "edgecolor": "black",
        "zorder": 3
    },
    medianprops={
        "color": "white",
        "linewidth": 2
    },
    whiskerprops={"color": "black"},
    capprops={"color": "black"},
    showfliers=False,
    ax=axes[1]
)

axes[1].set_xlabel("")
axes[1].set_ylabel("")
axes[1].set_title("RMSD Distribution Comparison (DiffDock)")

stat2, p2 = mannwhitneyu(rmsd, rmsd_mis, alternative="two-sided")
fig.text(
    0.005, 1.02, "(a)",
    fontsize=26,
    fontweight="bold",
    va="top",
    ha="left"
)
# plt.suptitle("RMSD Distribution Comparison (AlphaFold3)", y=1.02)
plt.tight_layout()
plt.show()


# Print statistics
print("Dataset 1:")
print("Matched Mean:", np.mean(rmsd_A))
print("Matched Median:", np.median(rmsd_A))
print("Mismatched Mean:", np.mean(rmsd_mis_A))
print("Mismatched Median:", np.median(rmsd_mis_A))
print("Mann–Whitney p-value:", p1)

print("\nDataset 2:")
print("Matched Mean:", np.mean(rmsd))
print("Matched Median:", np.median(rmsd))
print("Mismatched Mean:", np.mean(rmsd_mis))
print("Mismatched Median:", np.median(rmsd_mis))
print("Mann–Whitney p-value:", p2)
